In [1]:
# Importação das bibliotecas
import sqlite3
import pandas as pd

In [2]:
# Conectar a base de dados
conn = sqlite3.connect("BaseDados.db")

In [3]:
# Criação do cursor para a execução dos comandos SQL
cur = conn.cursor()

In [4]:
# Buscando todas as informações da tabela
resultado = cur.execute("SELECT * FROM dados").fetchall()

In [5]:
# Pegando o nome das colunas
colunas = [i[0] for i in cur.description]

In [6]:
# Salvando em um DataFrame do Pandas
df = pd.DataFrame(resultado, columns=colunas)

In [7]:
df.head()

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,1,1,9.0
1,1,2,Bárbara da Cunha,63546,barbara@hashtag.com,0,1,2,1,1,10.0
2,2,3,Kevin Melo,80515,kevin@hashtag.com,1,1,3,1,1,7.0
3,3,4,Pedro Henrique da Costa,68004,pedro@hashtag.com,1,1,1,1,1,4.0
4,4,5,Mirella Viana,28421,mirella@hashtag.com,1,1,1,1,1,7.0


In [8]:
df.shape

(43, 11)

In [9]:
# Selecione apenas as colunas nome_aluno e cod_matricula
df = pd.DataFrame(
    cur.execute("SELECT nome_aluno, cod_matricula FROM dados").fetchall(),
    columns=[i[0] for i in cur.description])
df.head()

,nome_aluno,cod_matricula
0,Maria Eduarda da Rocha,38273
1,Bárbara da Cunha,63546
2,Kevin Melo,80515
3,Pedro Henrique da Costa,68004
4,Mirella Viana,28421


In [10]:
# Selecione apenas as colunas nome_aluno e e-mail
df = pd.DataFrame(
    cur.execute("SELECT nome_aluno, [e-mail] as email FROM dados").fetchall(),
    columns=[i[0] for i in cur.description])
df.head()

,nome_aluno,email
0,Maria Eduarda da Rocha,maria@hashtag.com
1,Bárbara da Cunha,barbara@hashtag.com
2,Kevin Melo,kevin@hashtag.com
3,Pedro Henrique da Costa,pedro@hashtag.com
4,Mirella Viana,mirella@hashtag.com


In [11]:
# Criando uma função para otimizar a execução de vários comandos
def executa_sql(comando: str) -> pd.DataFrame:
    return pd.DataFrame(cur.execute(comando).fetchall(), columns=[i[0] for i in cur.description])

In [12]:
executa_sql("SELECT nome_aluno, [e-mail] as email FROM dados").tail()

,nome_aluno,email
38,Bárbara Freitas,barbara@hashtag.com
39,Melissa Ribeiro,melissa@hashtag.com
40,Maria Eduarda da Rocha,maria@hashtag.com
41,Eloah Aragão,eloah@hashtag.com
42,Eloah Aragão,eloah@hashtag.com


### Filtrando a base de dados

In [13]:
executa_sql("SELECT * FROM dados").head()

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,1,1,9.0
1,1,2,Bárbara da Cunha,63546,barbara@hashtag.com,0,1,2,1,1,10.0
2,2,3,Kevin Melo,80515,kevin@hashtag.com,1,1,3,1,1,7.0
3,3,4,Pedro Henrique da Costa,68004,pedro@hashtag.com,1,1,1,1,1,4.0
4,4,5,Mirella Viana,28421,mirella@hashtag.com,1,1,1,1,1,7.0


In [14]:
# Selecionando apenas os valores únicos (distintos) e retornando apenas 2 colunas (nome_aluno e cod_matricula)
df = executa_sql("SELECT * FROM dados")
df.loc[:,["nome_aluno","cod_matricula"]].drop_duplicates()

,nome_aluno,cod_matricula
0,Maria Eduarda da Rocha,38273
1,Bárbara da Cunha,63546
2,Kevin Melo,80515
3,Pedro Henrique da Costa,68004
4,Mirella Viana,28421
5,Lívia Jesus,22284
6,Lara Lopes,38362
7,Lucca Cardoso,45109
8,Isabelly Souza,31859
9,Cauã Porto,42674


In [15]:
# Agora com o comando SQL
executa_sql("SELECT DISTINCT nome_aluno, cod_matricula FROM dados")

,nome_aluno,cod_matricula
0,Maria Eduarda da Rocha,38273
1,Bárbara da Cunha,63546
2,Kevin Melo,80515
3,Pedro Henrique da Costa,68004
4,Mirella Viana,28421
5,Lívia Jesus,22284
6,Lara Lopes,38362
7,Lucca Cardoso,45109
8,Isabelly Souza,31859
9,Cauã Porto,42674


In [16]:
# Filtrando apenas alunos que tiveram problemas de acesso usando pandas
df = executa_sql("SELECT * FROM dados")
df.loc[df["acesso_plataforma"] == 0,["nome_aluno", "cod_matricula"]].drop_duplicates().reset_index(drop=True)

,nome_aluno,cod_matricula
0,Maria Eduarda da Rocha,38273
1,Bárbara da Cunha,63546
2,Lívia Jesus,22284
3,Cauã Porto,42674
4,Maria Ferreira,86563
5,Antônio Azevedo,29022
6,Francisco Pires,59518
7,Bárbara Freitas,19442


In [17]:
# Realizando o mesmo filtro com SQL
executa_sql("SELECT DISTINCT nome_aluno, cod_matricula FROM dados WHERE acesso_plataforma = 0")

,nome_aluno,cod_matricula
0,Maria Eduarda da Rocha,38273
1,Bárbara da Cunha,63546
2,Lívia Jesus,22284
3,Cauã Porto,42674
4,Maria Ferreira,86563
5,Antônio Azevedo,29022
6,Francisco Pires,59518
7,Bárbara Freitas,19442


In [18]:
# Verificando se existe algum aluno com problema de acesso E que entrou nos últimos 2 dias em SQL
executa_sql("SELECT * FROM dados WHERE acesso_plataforma = 0 AND dias_ultimo_acesso < 3 \
             AND rowid IN (SELECT rowid FROM dados WHERE acesso_plataforma = 0 AND dias_ultimo_acesso < 3 \
                           GROUP BY nome_aluno)")

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,1,2,Bárbara da Cunha,63546,barbara@hashtag.com,0,1,2,1,1,10.0
1,9,10,Cauã Porto,42674,cauã@hashtag.com,0,1,2,1,0,NaN
2,10,11,Maria Ferreira,86563,maria@hashtag.com,0,1,1,1,1,7.0


In [19]:
# Verificando se existe algum aluno com problema de acesso E que entrou nos últimos 2 dias em Pandas
df = executa_sql("SELECT * FROM dados")
df.loc[(df["acesso_plataforma"] == 0) & (df["dias_ultimo_acesso"] < 3),:].drop_duplicates(subset=["nome_aluno"]).reset_index(drop=True)

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,1,2,Bárbara da Cunha,63546,barbara@hashtag.com,0,1,2,1,1,10.0
1,9,10,Cauã Porto,42674,cauã@hashtag.com,0,1,2,1,0,NaN
2,10,11,Maria Ferreira,86563,maria@hashtag.com,0,1,1,1,1,7.0


In [20]:
# Verificando se existe algum aluno com problema de acesso OU acesso acima de 10 dias em SQL
executa_sql("SELECT * FROM dados WHERE (acesso_plataforma = 0 OR dias_ultimo_acesso > 10) \
            AND rowid IN (SELECT rowid FROM dados WHERE acesso_plataforma = 0 OR dias_ultimo_acesso > 10 \
                          GROUP BY nome_aluno)")

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,1,1,9.0
1,1,2,Bárbara da Cunha,63546,barbara@hashtag.com,0,1,2,1,1,10.0
2,5,6,Lívia Jesus,22284,lívia@hashtag.com,0,1,14,1,1,8.0
3,9,10,Cauã Porto,42674,cauã@hashtag.com,0,1,2,1,0,NaN
4,10,11,Maria Ferreira,86563,maria@hashtag.com,0,1,1,1,1,7.0
5,12,13,Antônio Azevedo,29022,lovedogs@hashtag.com,0,1,14,1,1,6.0
6,16,17,Francisco Pires,59518,x!c0@hashtag.com,0,1,13,1,0,NaN
7,17,18,Bárbara Freitas,19442,barbara@hashtag.com,0,1,12,1,1,6.0


In [21]:
# Verificando se existe algum aluno com problema de acesso OU acesso acima de 10 dias em Pandas
df = executa_sql("SELECT * FROM dados")
df.loc[(df["acesso_plataforma"] == 0) | (df["dias_ultimo_acesso"] > 10), :].drop_duplicates(subset="nome_aluno").reset_index(drop=True)

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,1,1,9.0
1,1,2,Bárbara da Cunha,63546,barbara@hashtag.com,0,1,2,1,1,10.0
2,5,6,Lívia Jesus,22284,lívia@hashtag.com,0,1,14,1,1,8.0
3,9,10,Cauã Porto,42674,cauã@hashtag.com,0,1,2,1,0,NaN
4,10,11,Maria Ferreira,86563,maria@hashtag.com,0,1,1,1,1,7.0
5,12,13,Antônio Azevedo,29022,lovedogs@hashtag.com,0,1,14,1,1,6.0
6,16,17,Francisco Pires,59518,x!c0@hashtag.com,0,1,13,1,0,NaN
7,17,18,Bárbara Freitas,19442,barbara@hashtag.com,0,1,12,1,1,6.0


In [22]:
# Buscando alunos com mais de 12 dias de acesso em SQL
executa_sql("SELECT * FROM dados WHERE dias_ultimo_acesso >= 12 \
             AND rowid IN (SELECT rowid FROM dados WHERE dias_ultimo_acesso >= 12 GROUP BY nome_aluno)")

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,1,1,9.0
1,5,6,Lívia Jesus,22284,lívia@hashtag.com,0,1,14,1,1,8.0
2,12,13,Antônio Azevedo,29022,lovedogs@hashtag.com,0,1,14,1,1,6.0
3,16,17,Francisco Pires,59518,x!c0@hashtag.com,0,1,13,1,0,NaN
4,17,18,Bárbara Freitas,19442,barbara@hashtag.com,0,1,12,1,1,6.0


In [23]:
# Buscando alunos com mais de 12 dias de acesso em Pandas
df = executa_sql("SELECT * FROM dados")
df.loc[df["dias_ultimo_acesso"] >= 12, :].drop_duplicates(subset="nome_aluno").reset_index(drop=True)

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,1,1,9.0
1,5,6,Lívia Jesus,22284,lívia@hashtag.com,0,1,14,1,1,8.0
2,12,13,Antônio Azevedo,29022,lovedogs@hashtag.com,0,1,14,1,1,6.0
3,16,17,Francisco Pires,59518,x!c0@hashtag.com,0,1,13,1,0,NaN
4,17,18,Bárbara Freitas,19442,barbara@hashtag.com,0,1,12,1,1,6.0


In [24]:
# Buscando alunos com mais de 12 dias de acesso em SQL usando o NOT
executa_sql("SELECT * FROM dados WHERE NOT dias_ultimo_acesso >= 12 \
             AND rowid IN (SELECT rowid FROM dados WHERE NOT dias_ultimo_acesso >= 12 GROUP BY nome_aluno)")

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,1,2,Bárbara da Cunha,63546,barbara@hashtag.com,0,1,2,1,1,10.0
1,2,3,Kevin Melo,80515,kevin@hashtag.com,1,1,3,1,1,7.0
2,3,4,Pedro Henrique da Costa,68004,pedro@hashtag.com,1,1,1,1,1,4.0
3,4,5,Mirella Viana,28421,mirella@hashtag.com,1,1,1,1,1,7.0
4,6,7,Lara Lopes,38362,lara@hashtag.com,1,1,1,1,0,NaN
5,7,8,Lucca Cardoso,45109,lucca@hashtag.com,1,1,3,1,0,NaN
6,8,9,Isabelly Souza,31859,isabelly@hashtag.com,1,1,3,1,1,7.0
7,9,10,Cauã Porto,42674,cauã@hashtag.com,0,1,2,1,0,NaN
8,10,11,Maria Ferreira,86563,maria@hashtag.com,0,1,1,1,1,7.0
9,11,12,Júlia Pinto,47086,julia@hashtag.com,1,1,2,1,1,6.0


In [25]:
# Confirmando o resultado acima retirando o NOT e trocando a condição booleana de >= para <
executa_sql("SELECT * FROM dados WHERE dias_ultimo_acesso < 12 \
             AND rowid IN (SELECT rowid FROM dados WHERE dias_ultimo_acesso < 12 GROUP BY nome_aluno)")

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,1,2,Bárbara da Cunha,63546,barbara@hashtag.com,0,1,2,1,1,10.0
1,2,3,Kevin Melo,80515,kevin@hashtag.com,1,1,3,1,1,7.0
2,3,4,Pedro Henrique da Costa,68004,pedro@hashtag.com,1,1,1,1,1,4.0
3,4,5,Mirella Viana,28421,mirella@hashtag.com,1,1,1,1,1,7.0
4,6,7,Lara Lopes,38362,lara@hashtag.com,1,1,1,1,0,NaN
5,7,8,Lucca Cardoso,45109,lucca@hashtag.com,1,1,3,1,0,NaN
6,8,9,Isabelly Souza,31859,isabelly@hashtag.com,1,1,3,1,1,7.0
7,9,10,Cauã Porto,42674,cauã@hashtag.com,0,1,2,1,0,NaN
8,10,11,Maria Ferreira,86563,maria@hashtag.com,0,1,1,1,1,7.0
9,11,12,Júlia Pinto,47086,julia@hashtag.com,1,1,2,1,1,6.0


In [26]:
# Retornando o mínimo, o máximo e a média da coluna último acesso
executa_sql(
    "SELECT \
        MIN(dias_ultimo_acesso) as Mínimo,\
        MAX(dias_ultimo_acesso) as Máximo,\
        ROUND(AVG(dias_ultimo_acesso),1) as Média\
    FROM dados")

,Mínimo,Máximo,Média
0,1,14,5.1


In [27]:
# Fazer um GroupBy com o pandas para agrupar por nome do aluno e contar quantas vezes ele aparece na base
df = executa_sql("SELECT * FROM dados")
df.groupby("nome_aluno")["id_aluno"].count()

nome_aluno
Antônio Azevedo            2
Bárbara Freitas            3
Bárbara da Cunha           2
Cauã Porto                 1
Eloah Aragão               5
Francisco Pires            1
Gabriela Costela           2
Isabelly Souza             2
Júlia Pinto                3
Kevin Melo                 2
Lara Lopes                 1
Laura Melo                 2
Lavínia Carvalho           1
Lucca Cardoso              1
Lívia Jesus                2
Maria Eduarda da Rocha     4
Maria Ferreira             3
Melissa Ribeiro            3
Mirella Viana              2
Pedro Henrique da Costa    1
Name: id_aluno, dtype: int64

In [28]:
# Agora realizar em SQL
executa_sql("SELECT nome_aluno, COUNT(id_aluno) as '' FROM dados GROUP BY nome_aluno")

,nome_aluno,
0,Antônio Azevedo,2
1,Bárbara Freitas,3
2,Bárbara da Cunha,2
3,Cauã Porto,1
4,Eloah Aragão,5
5,Francisco Pires,1
6,Gabriela Costela,2
7,Isabelly Souza,2
8,Júlia Pinto,3
9,Kevin Melo,2


In [29]:
# Fazendo para calcular a média das provas por aluno em SQL
executa_sql("SELECT\
                nome_aluno, ROUND(AVG(nota_prova),1) as Média \
             FROM dados WHERE nota_prova IS NOT NULL \
             GROUP BY nome_aluno")

,nome_aluno,Média
0,Antônio Azevedo,6.0
1,Bárbara Freitas,8.0
2,Bárbara da Cunha,10.0
3,Eloah Aragão,7.5
4,Gabriela Costela,8.0
5,Isabelly Souza,7.0
6,Júlia Pinto,8.0
7,Kevin Melo,7.0
8,Laura Melo,6.0
9,Lívia Jesus,8.0


In [30]:
# Fazendo para calcular a média das provas por aluno em SQL ordenando de forma decrescente
executa_sql("SELECT\
                nome_aluno, ROUND(AVG(nota_prova),1) as Média \
             FROM dados WHERE nota_prova IS NOT NULL \
             GROUP BY nome_aluno ORDER BY Média DESC")

,nome_aluno,Média
0,Bárbara da Cunha,10.0
1,Melissa Ribeiro,8.0
2,Lívia Jesus,8.0
3,Júlia Pinto,8.0
4,Gabriela Costela,8.0
5,Bárbara Freitas,8.0
6,Maria Eduarda da Rocha,7.7
7,Eloah Aragão,7.5
8,Mirella Viana,7.0
9,Kevin Melo,7.0


In [31]:
# Realizar o Group By retornando apenas os alunos com a média maior que 5 com pandas
df = executa_sql("SELECT * FROM dados")
df.groupby(["nome_aluno"])["nota_prova"].mean().round(1).dropna().loc[lambda x: x > 5].sort_values(ascending=False)

nome_aluno
Bárbara da Cunha          10.0
Bárbara Freitas            8.0
Gabriela Costela           8.0
Melissa Ribeiro            8.0
Lívia Jesus                8.0
Júlia Pinto                8.0
Maria Eduarda da Rocha     7.7
Eloah Aragão               7.5
Mirella Viana              7.0
Isabelly Souza             7.0
Kevin Melo                 7.0
Maria Ferreira             6.5
Antônio Azevedo            6.0
Laura Melo                 6.0
Name: nota_prova, dtype: float64

In [32]:
# Realizar o Group By retornando apenas os alunos com a média maior que 5 com SQL
executa_sql("""SELECT 
            nome_aluno, ROUND(AVG(nota_prova),1) as Média
            FROM dados WHERE nota_prova IS NOT NULL
            GROUP BY nome_aluno HAVING Média > 5
            ORDER BY Média DESC""")

# Ao final da instrução SQL, posso usar o LIMIT #num para retornar quantas linhas eu desejar (ex: LIMIT 3)

,nome_aluno,Média
0,Bárbara da Cunha,10.0
1,Melissa Ribeiro,8.0
2,Lívia Jesus,8.0
3,Júlia Pinto,8.0
4,Gabriela Costela,8.0
5,Bárbara Freitas,8.0
6,Maria Eduarda da Rocha,7.7
7,Eloah Aragão,7.5
8,Mirella Viana,7.0
9,Kevin Melo,7.0


In [33]:
# Reutilizar a query acima sem filtrar pelas médias acima de 5, mas incluir a coluna situação (APROVADO ou REPROVADO)
# Usar a cláusula CASE

executa_sql("""SELECT 
            nome_aluno, ROUND(AVG(nota_prova),1) as Média,
            (CASE
                WHEN ROUND(AVG(nota_prova),1) >= 7 THEN "APROVADO"
                WHEN ROUND(AVG(nota_prova),1) < 5 THEN "REPROVADO"
                ELSE "EM RECUPERAÇÃO"
            END) as Situação
            FROM dados WHERE nota_prova IS NOT NULL
            GROUP BY nome_aluno
            ORDER BY Média DESC""")

,nome_aluno,Média,Situação
0,Bárbara da Cunha,10.0,APROVADO
1,Melissa Ribeiro,8.0,APROVADO
2,Lívia Jesus,8.0,APROVADO
3,Júlia Pinto,8.0,APROVADO
4,Gabriela Costela,8.0,APROVADO
5,Bárbara Freitas,8.0,APROVADO
6,Maria Eduarda da Rocha,7.7,APROVADO
7,Eloah Aragão,7.5,APROVADO
8,Mirella Viana,7.0,APROVADO
9,Kevin Melo,7.0,APROVADO


In [34]:
# Utilizando uma subquery
executa_sql("""SELECT nome_aluno, Média, 
                (CASE
                WHEN Média >= 7 THEN "APROVADO"
                WHEN Média THEN "REPROVADO"
                ELSE "EM RECUPERAÇÃO"
            END) as Situação FROM (SELECT 
            nome_aluno, ROUND(AVG(nota_prova),1) as Média
            FROM dados WHERE nota_prova IS NOT NULL
            GROUP BY nome_aluno
            ORDER BY Média DESC)""")

,nome_aluno,Média,Situação
0,Bárbara da Cunha,10.0,APROVADO
1,Melissa Ribeiro,8.0,APROVADO
2,Lívia Jesus,8.0,APROVADO
3,Júlia Pinto,8.0,APROVADO
4,Gabriela Costela,8.0,APROVADO
5,Bárbara Freitas,8.0,APROVADO
6,Maria Eduarda da Rocha,7.7,APROVADO
7,Eloah Aragão,7.5,APROVADO
8,Mirella Viana,7.0,APROVADO
9,Kevin Melo,7.0,APROVADO


In [35]:
# Verificando os dias de último acesso iguais a 10, 11 e 12 usando o WHERE + AND
executa_sql("SELECT * FROM dados WHERE dias_ultimo_acesso >= 10 AND dias_ultimo_acesso <= 12")

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,1,1,9.0
1,17,18,Bárbara Freitas,19442,barbara@hashtag.com,0,1,12,1,1,6.0
2,20,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,2,1,8.0
3,32,18,Bárbara Freitas,19442,barbara@hashtag.com,0,1,12,2,1,10.0
4,34,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,3,1,6.0
5,38,18,Bárbara Freitas,19442,barbara@hashtag.com,0,1,12,3,0,NaN
6,40,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,4,0,NaN


In [36]:
# Alterando para usar o IN
executa_sql("SELECT * FROM dados WHERE dias_ultimo_acesso IN (10, 11, 12)")

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,1,1,9.0
1,17,18,Bárbara Freitas,19442,barbara@hashtag.com,0,1,12,1,1,6.0
2,20,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,2,1,8.0
3,32,18,Bárbara Freitas,19442,barbara@hashtag.com,0,1,12,2,1,10.0
4,34,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,3,1,6.0
5,38,18,Bárbara Freitas,19442,barbara@hashtag.com,0,1,12,3,0,NaN
6,40,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,4,0,NaN


In [37]:
# Utilizando o IN para filtrar estas matrículas: 28421, 38273, 42674 e 65749
executa_sql("SELECT * FROM dados WHERE cod_matricula IN (28421, 38273, 42674, 65749) GROUP BY cod_matricula")

,index,id_aluno,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado,dias_ultimo_acesso,nr_prova,prova_feita,nota_prova
0,4,5,Mirella Viana,28421,mirella@hashtag.com,1,1,1,1,1,7.0
1,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,12,1,1,9.0
2,9,10,Cauã Porto,42674,cauã@hashtag.com,0,1,2,1,0,NaN
3,15,16,Eloah Aragão,65749,eloah@hashtag.com,1,1,3,1,1,8.0


In [38]:
# Verificando alunos que o nome seja igual Maria Ferreira utilizando IN
executa_sql("SELECT DISTINCT nome_aluno, cod_matricula FROM dados WHERE nome_aluno IN ('Maria Ferreira')")

,nome_aluno,cod_matricula
0,Maria Ferreira,86563


In [39]:
# Fazendo a mesma consulta utilizando o LIKE
executa_sql("SELECT DISTINCT nome_aluno, cod_matricula FROM dados WHERE nome_aluno LIKE 'Maria Ferreira'")

,nome_aluno,cod_matricula
0,Maria Ferreira,86563


In [40]:
# Verificando todas as alunas chamadas Maria
executa_sql("SELECT DISTINCT nome_aluno, cod_matricula FROM dados WHERE nome_aluno LIKE 'Maria%'")

,nome_aluno,cod_matricula
0,Maria Eduarda da Rocha,38273
1,Maria Ferreira,86563


In [41]:
# Verificando se o e-mail possui o caracter !
executa_sql("SELECT nome_aluno, cod_matricula, [e-mail] FROM dados WHERE [e-mail] LIKE '%!%'")

,nome_aluno,cod_matricula,e-mail
0,Francisco Pires,59518,x!c0@hashtag.com


In [42]:
# Verificando se possui !, ã ou í
executa_sql("""SELECT DISTINCT nome_aluno, cod_matricula, [e-mail], acesso_plataforma, acesso_liberado 
               FROM dados
               WHERE [e-mail] LIKE '%!%' OR [e-mail] LIKE '%ã%' OR [e-mail] LIKE '%í%'""")

,nome_aluno,cod_matricula,e-mail,acesso_plataforma,acesso_liberado
0,Lívia Jesus,22284,lívia@hashtag.com,0,1
1,Cauã Porto,42674,cauã@hashtag.com,0,1
2,Francisco Pires,59518,x!c0@hashtag.com,0,1


## Usando o JOIN
#### Nova base de dados com 3 tabelas diferentes

In [81]:
# Conectando na nova base de dados
conn2 = sqlite3.connect("BaseDados2.db")

In [82]:
# Criando o cursor para a execução dos comandos
cur2 = conn2.cursor() 

In [83]:
# Criando uma função para otimizar a execução de vários comandos
def executa_sql2(comando: str) -> pd.DataFrame:
    return pd.DataFrame(cur2.execute(comando).fetchall(), columns=[i[0] for i in cur2.description])

In [46]:
# Visualizando a tabela alunos
executa_sql2("SELECT * FROM alunos LIMIT 5")

,index,id_aluno,nome_aluno,cod_matricula,e-mail
0,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com
1,1,2,Bárbara da Cunha,63546,barbara@hashtag.com
2,2,3,Kevin Melo,80515,kevin@hashtag.com
3,3,4,Pedro Henrique da Costa,68004,pedro@hashtag.com
4,4,5,Mirella Viana,28421,mirella@hashtag.com


In [47]:
# Visualizando a tabela plataforma
executa_sql2("SELECT * FROM plataforma LIMIT 5")

,index,id_plataforma,cod_matricula,acesso_plataforma,acesso_liberado,dias_ultimo_acesso
0,0,1,38273,0,1,12
1,1,2,63546,0,1,2
2,2,3,80515,1,1,3
3,3,4,68004,1,1,1
4,4,5,28421,1,1,1


In [48]:
# Visualizando a tabela provas
executa_sql2("SELECT * FROM provas LIMIT 5")

,index,id_prova,cod_matricula,nr_prova,prova_feita,nota_prova
0,0,1,38273,1,1,9.0
1,1,2,63546,1,1,10.0
2,2,3,80515,1,1,7.0
3,3,4,68004,1,1,4.0
4,4,5,28421,1,1,7.0


In [52]:
# Unindo a tabela alunos com a tabela plataforma
df_alunos = executa_sql2("SELECT * FROM alunos")
df_plataforma = executa_sql2("SELECT * FROM plataforma")

df_merged = pd.merge(df_alunos, df_plataforma, how="left", on="cod_matricula")
df_merged

,index_x,id_aluno,nome_aluno,cod_matricula,e-mail,index_y,id_plataforma,acesso_plataforma,acesso_liberado,dias_ultimo_acesso
0,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,0,1,12
1,1,2,Bárbara da Cunha,63546,barbara@hashtag.com,1,2,0,1,2
2,2,3,Kevin Melo,80515,kevin@hashtag.com,2,3,1,1,3
3,3,4,Pedro Henrique da Costa,68004,pedro@hashtag.com,3,4,1,1,1
4,4,5,Mirella Viana,28421,mirella@hashtag.com,4,5,1,1,1
5,5,6,Lívia Jesus,22284,lívia@hashtag.com,5,6,0,1,14
6,6,7,Lara Lopes,38362,lara@hashtag.com,6,7,1,1,1
7,7,8,Lucca Cardoso,45109,lucca@hashtag.com,7,8,1,1,3
8,8,9,Isabelly Souza,31859,isabelly@hashtag.com,8,9,1,1,3
9,9,10,Cauã Porto,42674,cauã@hashtag.com,9,10,0,1,2


In [64]:
# Verificando se algum aluno não possui registro na tabela plataforma

# 1ª forma de fazer
df_merged.loc[df_merged["id_plataforma"].isnull(),:]

# 2ª forma de fazer
# df_merged[df_merged["id_plataforma"].isnull()]

# 3ª forma de fazer
# df_merged[df_merged.id_plataforma.isnull()]

,index_x,id_aluno,nome_aluno,cod_matricula,e-mail,index_y,id_plataforma,acesso_plataforma,acesso_liberado,dias_ultimo_acesso


### Usando o JOIN no SQL

In [66]:
# Trazendo as informações da tabela aluno com a tabela plataforma através do JOIN
executa_sql2("""
                SELECT * FROM alunos LEFT JOIN plataforma 
                ON alunos.cod_matricula = plataforma.cod_matricula
             """)

,index,id_aluno,nome_aluno,cod_matricula,e-mail,index,id_plataforma,cod_matricula,acesso_plataforma,acesso_liberado,dias_ultimo_acesso
0,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,38273,0,1,12
1,1,2,Bárbara da Cunha,63546,barbara@hashtag.com,1,2,63546,0,1,2
2,2,3,Kevin Melo,80515,kevin@hashtag.com,2,3,80515,1,1,3
3,3,4,Pedro Henrique da Costa,68004,pedro@hashtag.com,3,4,68004,1,1,1
4,4,5,Mirella Viana,28421,mirella@hashtag.com,4,5,28421,1,1,1
5,5,6,Lívia Jesus,22284,lívia@hashtag.com,5,6,22284,0,1,14
6,6,7,Lara Lopes,38362,lara@hashtag.com,6,7,38362,1,1,1
7,7,8,Lucca Cardoso,45109,lucca@hashtag.com,7,8,45109,1,1,3
8,8,9,Isabelly Souza,31859,isabelly@hashtag.com,8,9,31859,1,1,3
9,9,10,Cauã Porto,42674,cauã@hashtag.com,9,10,42674,0,1,2


In [76]:
# Trazendo as informações da tabela plataforma e fazendo o JOIN com a tabela alunos
executa_sql2("""
                SELECT * FROM alunos RIGHT JOIN plataforma
                ON alunos.cod_matricula = plataforma.cod_matricula
             """)

,index,id_aluno,nome_aluno,cod_matricula,e-mail,index,id_plataforma,cod_matricula,acesso_plataforma,acesso_liberado,dias_ultimo_acesso
0,0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com,0,1,38273,0,1,12
1,1,2,Bárbara da Cunha,63546,barbara@hashtag.com,1,2,63546,0,1,2
2,2,3,Kevin Melo,80515,kevin@hashtag.com,2,3,80515,1,1,3
3,3,4,Pedro Henrique da Costa,68004,pedro@hashtag.com,3,4,68004,1,1,1
4,4,5,Mirella Viana,28421,mirella@hashtag.com,4,5,28421,1,1,1
5,5,6,Lívia Jesus,22284,lívia@hashtag.com,5,6,22284,0,1,14
6,6,7,Lara Lopes,38362,lara@hashtag.com,6,7,38362,1,1,1
7,7,8,Lucca Cardoso,45109,lucca@hashtag.com,7,8,45109,1,1,3
8,8,9,Isabelly Souza,31859,isabelly@hashtag.com,8,9,31859,1,1,3
9,9,10,Cauã Porto,42674,cauã@hashtag.com,9,10,42674,0,1,2


In [79]:
executa_sql2("SELECT * FROM plataforma")

,index,id_plataforma,cod_matricula,acesso_plataforma,acesso_liberado,dias_ultimo_acesso
0,0,1,38273,0,1,12
1,1,2,63546,0,1,2
2,2,3,80515,1,1,3
3,3,4,68004,1,1,1
4,4,5,28421,1,1,1
5,5,6,22284,0,1,14
6,6,7,38362,1,1,1
7,7,8,45109,1,1,3
8,8,9,31859,1,1,3
9,9,10,42674,0,1,2


In [85]:
# Visualizando apenas os valores da tabela plataforma que não estão na tabela alunos
executa_sql2("""
                SELECT a.nome_aluno, a.[e-mail], p.acesso_plataforma, coalesce(a.cod_matricula,p.cod_matricula) as cod_matricula
                FROM plataforma p LEFT JOIN alunos a ON a.cod_matricula = p.cod_matricula 
                WHERE id_aluno IS NULL
             """)

,nome_aluno,e-mail,acesso_plataforma,cod_matricula
0,None,None,0,91047


### Trabalhando com UNION

In [91]:
# Conectando na nova base de dados
conn3 = sqlite3.connect("BaseDados3.db")

In [92]:
# Criando o cursor
cur3 = conn3.cursor()

In [93]:
# Criando uma função para otimizar a execução de vários comandos
def executa_sql3(comando: str) -> pd.DataFrame:
    return pd.DataFrame(cur3.execute(comando).fetchall(), columns=[i[0] for i in cur3.description])

In [94]:
# Conhecendo a nova tabela alunos_novos
executa_sql3("SELECT * FROM alunos_novos")

,id_aluno,nome_aluno,cod_matricula,e-mail
0,1,Pedro Carlos,115640,pedro@hashtag.com
1,2,Beatriz Rezende,100310,bia@hashtag.com
2,3,Sheila Costa,112366,sheila@hashtag.com
3,21,Jorge Henrique Gomes,89791,jorge@hashtag.com


In [95]:
# Unindo as tabelas alunos e alunos_novos com UNION
executa_sql3("SELECT * FROM alunos UNION SELECT * FROM alunos_novos")

,id_aluno,nome_aluno,cod_matricula,e-mail
0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com
1,1,Pedro Carlos,115640,pedro@hashtag.com
2,2,Beatriz Rezende,100310,bia@hashtag.com
3,2,Bárbara da Cunha,63546,barbara@hashtag.com
4,3,Kevin Melo,80515,kevin@hashtag.com
5,3,Sheila Costa,112366,sheila@hashtag.com
6,4,Pedro Henrique da Costa,68004,pedro@hashtag.com
7,5,Mirella Viana,28421,mirella@hashtag.com
8,6,Lívia Jesus,22284,lívia@hashtag.com
9,7,Lara Lopes,38362,lara@hashtag.com


In [96]:
# Com o UNION ALL as informações das 2 tabelas são mantidas, ou seja, haverá dados duplicados
executa_sql3("SELECT * FROM alunos UNION ALL SELECT * FROM alunos_novos")

,id_aluno,nome_aluno,cod_matricula,e-mail
0,1,Maria Eduarda da Rocha,38273,maria@hashtag.com
1,2,Bárbara da Cunha,63546,barbara@hashtag.com
2,3,Kevin Melo,80515,kevin@hashtag.com
3,4,Pedro Henrique da Costa,68004,pedro@hashtag.com
4,5,Mirella Viana,28421,mirella@hashtag.com
5,6,Lívia Jesus,22284,lívia@hashtag.com
6,7,Lara Lopes,38362,lara@hashtag.com
7,8,Lucca Cardoso,45109,lucca@hashtag.com
8,9,Isabelly Souza,31859,isabelly@hashtag.com
9,10,Cauã Porto,42674,cauã@hashtag.com


In [100]:
# Selecionar todos os valores da tabela alunos fazendo o JOIN com a tabela plataforma
executa_sql3("""
                SELECT 
                    a.nome_aluno, coalesce(a.cod_matricula, p.cod_matricula) as cod_matricula, a.[e-mail], p.acesso_plataforma
                FROM alunos a LEFT JOIN plataforma p
                ON a.cod_matricula = p.cod_matricula
             """)

,nome_aluno,cod_matricula,e-mail,acesso_plataforma
0,Maria Eduarda da Rocha,38273,maria@hashtag.com,0.0
1,Bárbara da Cunha,63546,barbara@hashtag.com,0.0
2,Kevin Melo,80515,kevin@hashtag.com,1.0
3,Pedro Henrique da Costa,68004,pedro@hashtag.com,1.0
4,Mirella Viana,28421,mirella@hashtag.com,1.0
5,Lívia Jesus,22284,lívia@hashtag.com,0.0
6,Lara Lopes,38362,lara@hashtag.com,1.0
7,Lucca Cardoso,45109,lucca@hashtag.com,1.0
8,Isabelly Souza,31859,isabelly@hashtag.com,1.0
9,Cauã Porto,42674,cauã@hashtag.com,0.0


In [101]:
# Selecionar todos os valores da tabela plataforma fazendo o JOIN com a tabela alunos
executa_sql3("""
                SELECT 
                    a.nome_aluno, coalesce(a.cod_matricula, p.cod_matricula) as cod_matricula, a.[e-mail], p.acesso_plataforma
                FROM plataforma p LEFT JOIN alunos a
                ON a.cod_matricula = p.cod_matricula
             """)

,nome_aluno,cod_matricula,e-mail,acesso_plataforma
0,Maria Eduarda da Rocha,38273,maria@hashtag.com,0
1,Bárbara da Cunha,63546,barbara@hashtag.com,0
2,Kevin Melo,80515,kevin@hashtag.com,1
3,Pedro Henrique da Costa,68004,pedro@hashtag.com,1
4,Mirella Viana,28421,mirella@hashtag.com,1
5,Lívia Jesus,22284,lívia@hashtag.com,0
6,Lara Lopes,38362,lara@hashtag.com,1
7,Lucca Cardoso,45109,lucca@hashtag.com,1
8,Isabelly Souza,31859,isabelly@hashtag.com,1
9,Cauã Porto,42674,cauã@hashtag.com,0


In [102]:
# Para fazer o FULL JOIN, podemos exatamente pegar esses dois LEFT JOINS e fazer o UNION
executa_sql3("""
                SELECT 
                    a.nome_aluno, coalesce(a.cod_matricula, p.cod_matricula) as cod_matricula, a.[e-mail], p.acesso_plataforma
                FROM plataforma p LEFT JOIN alunos a
                ON a.cod_matricula = p.cod_matricula
                UNION
                SELECT 
                    a.nome_aluno, coalesce(a.cod_matricula, p.cod_matricula) as cod_matricula, a.[e-mail], p.acesso_plataforma
                FROM alunos a LEFT JOIN plataforma p
                ON a.cod_matricula = p.cod_matricula
             """)

,nome_aluno,cod_matricula,e-mail,acesso_plataforma
0,NaN,91047,NaN,0.0
1,Antônio Azevedo,29022,lovedogs@hashtag.com,0.0
2,Bárbara Freitas,19442,barbara@hashtag.com,0.0
3,Bárbara da Cunha,63546,barbara@hashtag.com,0.0
4,Cauã Porto,42674,cauã@hashtag.com,0.0
5,Eloah Aragão,65749,eloah@hashtag.com,1.0
6,Francisco Pires,59518,x!c0@hashtag.com,0.0
7,Gabriela Costela,21262,gabriela@hashtag.com,1.0
8,Isabelly Souza,31859,isabelly@hashtag.com,1.0
9,Jorge Henrique Gomes,89791,jorge@hashtag.com,NaN
